# 0. Libraries

In [7]:
import sys
sys.path.append('...')
from src.flight_ontogeny import helpers, helpers_BW, Tracking
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import ListedColormap
import seaborn as sns
import scipy.signal as signal
from tqdm.notebook import  tqdm, tqdm_notebook

PackageNotFoundError: No package metadata was found for src.flight_ontogeny

In [ ]:
import os
import psutil

# Detect cores
physical_cores = psutil.cpu_count(logical=False)
logical_cores = psutil.cpu_count(logical=True)

print(f"Detected Physical cores: {physical_cores}, Logical cores: {logical_cores}")

# Set environment variables
os.environ["LOKY_MAX_CPU_COUNT"] = str(physical_cores)  # Use physical cores
os.environ["OMP_NUM_THREADS"] = "2"  # Recommended to avoid MKL memory leak

print("Environment variables set:")
print(f"LOKY_MAX_CPU_COUNT = {os.environ['LOKY_MAX_CPU_COUNT']}")
print(f"OMP_NUM_THREADS = {os.environ['OMP_NUM_THREADS']}")

# Now you can safely import joblib or sklearn without warnings

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="KMeans is known to have a memory leak")

In [ ]:
from importlib import reload 
reload(helpers)
reload(helpers_BW)
reload(Tracking)

# I. Reviewing all the force graphs 
1. Total 1043 force graphs 
2. Plots with the mix of landing + takeoff/ two peaks not well detected for bodyweight and impulse. 
3. Have to clip out OR divide into multiple chunks
4. Best to do manually to ensure the most number of samples as we only have 1043 take-offs. 

## Protocol
1. Load the dataset df
2. Review in the 11 chunks of 100, in 25 x 4 grid. 
3. For each chunk, apply the changes. 

## 1. Load the dataset

In [ ]:
path = "C:\\Users\\kmh\\Documents\\DATA\\zebras_2025\\FT_dataframe_subset_golden8_butterfilt.csv"
df = pd.read_csv(path, index_col=0,
                 dtype = {'FlightID': int, 'Bird': str, 'Age': int, 
                          'Landing': pd.Int64Dtype(), 'Note': str,'Frame': float, 
                          'Fx': float,'Fy': float,'Fz': float,'Tx': float,'Ty': float,'Tz': float})
df

In [ ]:
df['Ftotal_filt'] = np.sqrt(df['Fx_filt']**2+df['Fy_filt']**2+df['Fz_filt']**2)

## 2. Make 11 chunks of 100 flights 

In [ ]:
flight_df = helpers_BW.make_flight_df(df)
flight_df 

In [ ]:
# return flight_df1 ... flight_df11 with 100 rows each (last one with 43 rows)
# have no function so do it manually for now.
# make sure no flights overlap.
flight_df1 = flight_df.iloc[0:100,:]
flight_df2 = flight_df.iloc[100:200,:]
flight_df3 = flight_df.iloc[200:300,:]
flight_df4 = flight_df.iloc[300:400,:]
flight_df5 = flight_df.iloc[400:500,:]
flight_df6 = flight_df.iloc[500:600,:]
flight_df7 = flight_df.iloc[600:700,:]
flight_df8 = flight_df.iloc[700:800,:]
flight_df9 = flight_df.iloc[800:900,:]
flight_df10 = flight_df.iloc[900:1000,:]
flight_df11 = flight_df.iloc[1000:,:]
# learned that when doing iloc[a:b,:], bth row is excluded. 



## 3. make graphs

### chunk 1

#### pre-edit plots

In [ ]:
# show the plots of Fy for each flight in 4 columns, 25 rows.

fig, axs = plt.subplots(nrows=25, ncols=4, figsize=(20, 120))
axs = axs.flatten()

for i, flight in enumerate(flight_df1['FlightID'].unique()):
    trial_data = df[df['FlightID'] == flight]

    age = trial_data[trial_data['FlightID'] == flight]['Age'].values[0]
    bird = trial_data[trial_data['FlightID'] == flight]['Bird'].values[0]
    takeoff = trial_data[trial_data['FlightID'] == flight]['Takeoff'].values[0]

    time = trial_data['Time'].values
    Ftotal_filt = trial_data['Ftotal_filt'].values

    #detect plateaus and bodyweight  
    plateaus, bodyweight = helpers_BW.zbdetect_bw_Ftot(trial_data)

    #overlay 'Fz' by 'Time' in yellow line
    axs[i].plot(time, trial_data['Fz_filt'], label='Fz_filt', color='yellow')
    
    #overlay 'Fy' by 'Time' in blue line
    axs[i].plot(time, trial_data['Fy_filt'], label='Fy_filt')

    #overlay 'Fx' by 'Time'in orange line
    axs[i].plot(time, trial_data['Fx_filt'], label='Fx_filt', color='orange')

    #overlay 'Ftotal_filt'by 'Time' in pink line
    axs[i].plot(time, trial_data['Ftotal_filt'], label='Ftotal_filt', color='pink')

    # Highlight plateau regions in yellow
    for start_idx, end_idx in plateaus:
        axs[i].axvspan(time[start_idx], time[end_idx], color='yellow', alpha=0.3)

    # Plot bodyweight as horizontal dashed line
    if not np.isnan(bodyweight):
        axs[i].axhline(y=bodyweight, color='grey', linestyle='--', label='Bodyweight')
    
    #axs[i].axhline(y=helpers_BW.get_bodyweight(df, flight, use_filt=True), color='r', linestyle='--', label='Bodyweight')
    # Show bird, age, takeoff in title
    axs[i].set_title(f'FlightID: {flight}_{bird}_{age}_{takeoff}')
    axs[i].set_xlabel('Time (s)')
    axs[i].set_ylabel('F (N)')
    axs[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
df.dtypes

In [ ]:
print(df.shape)
# make a list of flight ID to filter out 
flights_to_exclude = [2304001, 2304003, 2304005, 2102401, 2103206, 2104001]
# save the flights to edit first
df1_to_edit = df[df['FlightID'].isin(flights_to_exclude)]

# exclude from original dataset 
df = df[~df['FlightID'].isin(flights_to_exclude)
                        ]
print(df.shape) 

In [ ]:
df1_to_edit.shape

In [ ]:
# for each flightID in df1_to_edit, make a subset with the correct time range

for flightID in df1_to_edit['FlightID'].unique():

    if flightID == 2304001:
        #make subset of this flightID and save the rows with Time 4-8s
        subset1 = df1_to_edit[(df1_to_edit['FlightID'] == flightID) & (df1_to_edit['Time'] >= 4) & (df1_to_edit['Time'] <= 8)]

    elif flightID == 2304003:

        subset2 = df1_to_edit[(df1_to_edit['FlightID'] == flightID) & (df1_to_edit['Time'] >= 4) & (df1_to_edit['Time'] <= 8)]

    elif flightID == 2304005:
        subset3 = df1_to_edit[(df1_to_edit['FlightID'] == flightID) & (df1_to_edit['Time'] >= 3.5) & (df1_to_edit['Time'] <= 7.5)]

    elif flightID == 2102401:
        subset4 = df1_to_edit[(df1_to_edit['FlightID'] == flightID) & (df1_to_edit['Time'] >= -1.0) & (df1_to_edit['Time'] <= 1.0)]
    
    elif flightID == 2103206:
        subset5 = df1_to_edit[(df1_to_edit['FlightID'] == flightID) & (df1_to_edit['Time'] >= 0) & (df1_to_edit['Time'] <= 4)]

    elif flightID == 2104001: # flightID == 214001
        subset6 = df1_to_edit[(df1_to_edit['FlightID'] == flightID) & (df1_to_edit['Time'] >= 2) & (df1_to_edit['Time'] <= 4)]

# concatenate all the subsets
df1_edited = pd.concat([subset1, subset2, subset3, subset4, subset5, subset6])

# concatenate back to flight_df1
df = pd.concat([df, df1_edited])

#### post-edit plots

In [ ]:
# show the plots of Fy for each flight in 4 columns, 25 rows.

fig, axs = plt.subplots(nrows=25, ncols=4, figsize=(20, 120))
axs = axs.flatten()

for i, flight in enumerate(flight_df1['FlightID'].unique()):
    trial_data = df[df['FlightID'] == flight]

    age = trial_data[trial_data['FlightID'] == flight]['Age'].values[0]
    bird = trial_data[trial_data['FlightID'] == flight]['Bird'].values[0]
    takeoff = trial_data[trial_data['FlightID'] == flight]['Takeoff'].values[0]

    time = trial_data['Time'].values
    Ftotal_filt = trial_data['Ftotal_filt'].values

    #detect plateaus and bodyweight  
    plateaus, bodyweight = helpers_BW.zbdetect_bw_Ftot(trial_data)

    #overlay 'Fz' by 'Time' in yellow line
    axs[i].plot(time, trial_data['Fz_filt'], label='Fz_filt', color='yellow')
    
    #overlay 'Fy' by 'Time' in blue line
    axs[i].plot(time, trial_data['Fy_filt'], label='Fy_filt')

    #overlay 'Fx' by 'Time'in orange line
    axs[i].plot(time, trial_data['Fx_filt'], label='Fx_filt', color='orange')

    #overlay 'Ftotal_filt'by 'Time' in pink line
    axs[i].plot(time, trial_data['Ftotal_filt'], label='Ftotal_filt', color='pink')

    # Highlight plateau regions in yellow
    for start_idx, end_idx in plateaus:
        axs[i].axvspan(time[start_idx], time[end_idx], color='yellow', alpha=0.3)

    # Plot bodyweight as horizontal dashed line
    if not np.isnan(bodyweight):
        axs[i].axhline(y=bodyweight, color='grey', linestyle='--', label='Bodyweight')
    
    #axs[i].axhline(y=helpers_BW.get_bodyweight(df, flight, use_filt=True), color='r', linestyle='--', label='Bodyweight')
    # Show bird, age, takeoff in title
    axs[i].set_title(f'FlightID: {flight}_{bird}_{age}_{takeoff}')
    axs[i].set_xlabel('Time (s)')
    axs[i].set_ylabel('F (N)')
    axs[i].legend()

plt.tight_layout()
plt.show()

### chunk 2

#### pre-edit

In [ ]:
# show the plots of Fy for each flight in 4 columns, 25 rows.

helpers_BW.plot_flight_data(flight_df2, df)

In [ ]:
print(df.shape)
# make a list of flight ID to filter out 
flights_to_exclude = [3202404, 3202405, 3202806, 3208003, 3702402, 3702407]
#delete entirely 
# save the flights to edit first
df2_to_edit = df[df['FlightID'].isin(flights_to_exclude)]

# exclude from original dataset 
df = df[~df['FlightID'].isin(flights_to_exclude)
                        ]
print(df.shape) 

In [ ]:
df2_to_edit.shape

In [ ]:
# for each flightID in df2_to_edit, make a subset with the correct time range

for flightID in df2_to_edit['FlightID'].unique():

    if flightID == 3202404 or flightID == 3208003 or flightID == 3702402:
        continue

    elif flightID == 3202405:
        
        subset1 = df2_to_edit[(df2_to_edit['FlightID'] == flightID) & (df2_to_edit['Time'] >= -2.0) & (df2_to_edit['Time'] <= -0.5)]

    elif flightID == 3202806:
        subset2 = df2_to_edit[(df2_to_edit['FlightID'] == flightID) & (df2_to_edit['Time'] >= 0.5) & (df2_to_edit['Time'] <= 2.0)]

    elif flightID == 3702407:
        subset3 = df2_to_edit[(df2_to_edit['FlightID'] == flightID) & (df2_to_edit['Time'] >= -1.0) & (df2_to_edit['Time'] <= 1.0)]
    
    
# concatenate all the subsets
df2_edited = pd.concat([subset1, subset2, subset3])

# concatenate back to flight_df2
df = pd.concat([df, df2_edited])

In [ ]:
df.to_csv("C:\\Users\\kmh\\Documents\\DATA\\zebras_2025\\df_chunk2_edited.csv")
# up to chunk 2 edited

#### post-edit

In [ ]:
helpers_BW.plot_flight_data(flight_df2, df)

### chunk 3

#### pre-edit 

In [ ]:
helpers_BW.plot_flight_data(flight_df3, df)

In [ ]:
print(df.shape)
# make a list of flight ID to filter out 
flights_to_exclude = [2703601, 2704003, 2704004, 2704005, 2704006 ]
#delete entirely 
# save the flights to edit first
df3_to_edit = df[df['FlightID'].isin(flights_to_exclude)]

# exclude from original dataset 
df = df[~df['FlightID'].isin(flights_to_exclude)
                        ]
print(df.shape) 

In [ ]:
df3_to_edit.shape

In [ ]:
# for each flightID in df3_to_edit, make a subset with the correct time range

for flightID in df3_to_edit['FlightID'].unique():

    if flightID == 2703601 or flightID == 2704004:
        continue

    elif flightID == 2704003:
        
        subset1 = df3_to_edit[(df3_to_edit['FlightID'] == flightID) & (df3_to_edit['Time'] >= 0.5) & (df3_to_edit['Time'] <= 3.0)]

    elif flightID == 2704005:
        subset2 = df3_to_edit[(df3_to_edit['FlightID'] == flightID) & (df3_to_edit['Time'] >= 0.5) & (df3_to_edit['Time'] <= 3.0)]

    elif flightID == 2704006:
        subset3 = df3_to_edit[(df3_to_edit['FlightID'] == flightID) & (df3_to_edit['Time'] >= 0.5) & (df3_to_edit['Time'] <= 3.0)]
    
    
# concatenate all the subsets
df3_edited = pd.concat([subset1, subset2, subset3])

# concatenate back to flight_df3
df = pd.concat([df, df3_edited])

#### post-edit

In [ ]:
helpers_BW.plot_flight_data(flight_df3, df)

### chunk 4 

#### pre-edit

In [ ]:
helpers_BW.plot_flight_data(flight_df4, df)
# showing df graphs with the list in flight_df4

In [ ]:
print(df.shape)
# make a list of flight ID to filter out 
flights_to_exclude = [2710007, 2604005, 3002403 ]
#delete entirely 
# save the flights to edit first

df4_to_edit = df[df['FlightID'].isin(flights_to_exclude)]

# exclude from original dataset 
df = df[~df['FlightID'].isin(flights_to_exclude)
                        ]

print(df.shape) 
df4_to_edit.shape
# for each flightID in df4_to_edit, make a subset with the correct time range

for flightID in df4_to_edit['FlightID'].unique():

    if flightID == 2710007 or flightID == 3002403:
        continue

    elif flightID == 2604005:
        subset1 = df4_to_edit[(df4_to_edit['FlightID'] == flightID) & (df4_to_edit['Time'] >= -2.0) & (df4_to_edit['Time'] <= 1.0)]
        subset1.loc[:, 'FlightID'] = 2604005.1
        subset2 = df4_to_edit[(df4_to_edit['FlightID'] == flightID) & (df4_to_edit['Time'] >= 5.0)]
        subset2.loc[:, 'FlightID'] = 2604005.2

                              
# concatenate all the subsets
df4_edited = pd.concat([subset1, subset2])

# concatenate back to flight_df4
df = pd.concat([df, df4_edited])

#### post-edit

In [ ]:
#duplicate the row in flight_df4 with FlightID = 2604005 and replace one with 2604005.1 and one with 2604005.2 
flight_df4 = flight_df4.copy()
flight_df4.loc[flight_df4['FlightID'] == 2604005, 'FlightID'] = 2604005.1
new_row = flight_df4[flight_df4['FlightID'] == 2604005.1].copy()
new_row.loc[:, 'FlightID'] = 2604005.2
flight_df4 = pd.concat([flight_df4, new_row], ignore_index=True)
flight_df4['FlightID'].unique()

In [ ]:
flight_df4 = flight_df4[~flight_df4['FlightID'].isin(flights_to_exclude)]
# just to fit in to 25 x 4 grid.  

In [ ]:
flight_df4

In [ ]:
helpers_BW.plot_flight_data(flight_df4, df)

In [ ]:
df.to_csv("C:\\Users\\kmh\\Documents\\DATA\\zebras_2025\\df_upto_chunk4_edited.csv")

### chunk 5 

#### pre-edit

In [ ]:
helpers_BW.plot_flight_data(flight_df5,df)

In [ ]:
print(df.shape)
# make a list of flight ID to filter out 
flights_to_exclude = [1202005, 1202401, 1206001, 1206002, 1206003, 1206004, 1206005, 1206006, 1208001 ]
#delete entirely 
# save the flights to edit first

df5_to_edit = df[df['FlightID'].isin(flights_to_exclude)]

# exclude from original dataset 
df = df[~df['FlightID'].isin(flights_to_exclude)
                        ]

print(df.shape) 
df5_to_edit.shape
# for each flightID in df5_to_edit, make a subset with the correct time range

for flightID in df5_to_edit['FlightID'].unique():

    if flightID == 1202005 or flightID == 1202401:
        continue

    elif flightID == 1206001:
        subset1 = df5_to_edit[(df5_to_edit['FlightID'] == flightID) & (df5_to_edit['Time'] >= 2.0) & (df5_to_edit['Time'] <= 6.0)]

    elif flightID == 1206002:
        subset2 = df5_to_edit[(df5_to_edit['FlightID'] == flightID) & (df5_to_edit['Time'] <= 6.0)]

    elif flightID == 1206003:
        subset3 = df5_to_edit[(df5_to_edit['FlightID'] == flightID) & (df5_to_edit['Time'] <= 6.0)]

    elif flightID == 1206004:
        subset4 = df5_to_edit[(df5_to_edit['FlightID'] == flightID) & (df5_to_edit['Time'] <= 6.0)]
    
    elif flightID == 1206005:
        subset5 = df5_to_edit[(df5_to_edit['FlightID'] == flightID) & (df5_to_edit['Time'] <= 6.0)]
    
    elif flightID == 1206006:
        subset6 = df5_to_edit[(df5_to_edit['FlightID'] == flightID) & (df5_to_edit['Time'] <= 6.0)] 
    
    elif flightID == 1208001:
        subset7 = df5_to_edit[(df5_to_edit['FlightID'] == flightID) & (df5_to_edit['Time'] >= -1.0) & (df5_to_edit['Time'] <= 1.0)]


                              
# concatenate all the subsets
df5_edited = pd.concat([subset1, subset2, subset3, subset4, subset5, subset6, subset7])

# concatenate back to flight_df4
df = pd.concat([df, df5_edited])

#### post-edit

In [ ]:
helpers_BW.plot_flight_data(flight_df5, df)

all good! no additional edit

### chunk 6

#### pre-edit

In [ ]:
helpers_BW.plot_flight_data(flight_df6, df)

In [ ]:
print(df.shape)
# make a list of flight ID to filter out 
flights_to_exclude = [1308001, 1403201]
#delete entirely 
# save the flights to edit first

df6_to_edit = df[df['FlightID'].isin(flights_to_exclude)]

# exclude from original dataset 
df = df[~df['FlightID'].isin(flights_to_exclude)
                        ]

print(df.shape) 
df6_to_edit.shape
# for each flightID in df6_to_edit, make a subset with the correct time range

for flightID in df6_to_edit['FlightID'].unique():

    if flightID == 1308001:
        continue

    elif flightID == 1403201:
        subset1 = df6_to_edit[(df6_to_edit['FlightID'] == flightID) & (df6_to_edit['Time'] >= 0.0) & (df6_to_edit['Time'] <= 4.0)]

       
    

                              
# concatenate all the subsets
df6_edited = subset1

# concatenate back to flight_df4
df = pd.concat([df, df6_edited])

#### post-edit

In [ ]:
helpers_BW.plot_flight_data(flight_df6, df)

### chunk 7

#### pre-edit

In [ ]:
helpers_BW.plot_flight_data(flight_df7, df)

In [ ]:
print(df[df['FlightID'] == 1602001].shape)
print(df[df['FlightID'] == 1602003].shape)
print(df[df['FlightID'] == 1602004].shape)
print(df[df['FlightID'] == 1602401].shape)
print(df[df['FlightID'] == 1602402].shape)
print(df[df['FlightID'] == 1602403].shape)
print(df[df['FlightID'] == 1602404].shape)
print(df[df['FlightID'] == 1602405].shape)
print(df[df['FlightID'] == 1602801].shape)
print(df[df['FlightID'] == 1603201].shape)
print(df[df['FlightID'] == 1603202].shape)
print(df[df['FlightID'] == 1603601].shape)
print(df[df['FlightID'] == 1603602].shape)
print(df[df['FlightID'] == 1603603].shape)
print(df[df['FlightID'] == 1603604].shape)
print(df[df['FlightID'] == 1603605].shape)
# all the ones not graphed only have recordings of 0.007 seconds. 
# not sure why, could later check with the raw data. purpleR-blackL age 20, 24, 28, 32, 36 days.


In [ ]:
# for each flightID in df3_to_edit, make a subset with the correct time range
print(df.shape)
# make a list of flight ID to filter out 
flights_to_exclude = [1602001, 1602002, 1602003, 1602004, 
                      1602401, 1602402, 1602403, 1602404, 1602405, 
                      1602801, 1602802, 1603201, 1603202, 1603203, 1603204,
                      1603601, 1603602, 1603603, 1603604, 1603605, 
                      2202804, # all the flights being excluded - have v short data
                      1404001, 1404002, 1404003, # the ones to be trimmed
                      1404004, 1404005, 1406005, 1408003, 1408004, 1410004, 2202404
                      ]
#delete entirely 
# save the flights to edit first
df7_to_edit = df[df['FlightID'].isin(flights_to_exclude)]

# exclude from original dataset 
df = df[~df['FlightID'].isin(flights_to_exclude)
                        ]
print(df.shape) 


for flightID in df7_to_edit['FlightID'].unique():
    #21 flightIDs to exclude so only concatenating flights that are being trimmed
    if flightID == 1404001:
        subset1 = df7_to_edit[(df7_to_edit['FlightID'] == flightID) &  (df7_to_edit['Time'] <= 4.0)]
        subset1.loc[:, 'FlightID'] = 1404001.1
        subset2 = df7_to_edit[(df7_to_edit['FlightID'] == flightID) & (df7_to_edit['Time'] >= 10.0) & (df7_to_edit['Time'] <= 14.0)]
        subset2.loc[:, 'FlightID'] = 1404001.2
        
    elif flightID == 1404002:
        subset3 = df7_to_edit[(df7_to_edit['FlightID'] == flightID) & (df7_to_edit['Time'] >= 8.0) & (df7_to_edit['Time'] <= 12.0)]

    elif flightID == 1404003:
        subset4 = df7_to_edit[(df7_to_edit['FlightID'] == flightID) & (df7_to_edit['Time'] >= 8.0) & (df7_to_edit['Time'] <= 12.0)]

    elif flightID == 1404004:
        subset5 = df7_to_edit[(df7_to_edit['FlightID'] == flightID) & (df7_to_edit['Time'] >= 3.0) & (df7_to_edit['Time'] <= 7.0)]

    elif flightID == 1404005:

        subset6 = df7_to_edit[(df7_to_edit['FlightID'] == flightID) & (df7_to_edit['Time'] >= 2.0) & (df7_to_edit['Time'] <= 6.0)]

    elif flightID == 1406005:
        subset7 = df7_to_edit[(df7_to_edit['FlightID'] == flightID) & (df7_to_edit['Time'] >= -1.0) & (df7_to_edit['Time'] <= 2.0)]

    elif flightID == 1408003:
        subset8 = df7_to_edit[(df7_to_edit['FlightID'] == flightID) & (df7_to_edit['Time'] <= -1.0)]
    
    elif flightID == 1408004:
        subset9 = df7_to_edit[(df7_to_edit['FlightID'] == flightID) & (df7_to_edit['Time'] <= -1.0)]
    
    elif flightID == 1410004:
        subset10 = df7_to_edit[(df7_to_edit['FlightID'] == flightID) & (df7_to_edit['Time'] >= 0.5) & (df7_to_edit['Time'] <= 1.5)]
    
    elif flightID == 2202404:
        subset11 = df7_to_edit[(df7_to_edit['FlightID'] == flightID) & (df7_to_edit['Time'] >= 0.0) & (df7_to_edit['Time'] <= 4.0)]

# concatenate all the subsets
df7_edited = pd.concat([subset1, subset2, subset3, subset4, subset5, subset6, subset7, subset8, subset9, subset10, subset11])

# concatenate back to flight_df3
df = pd.concat([df, df7_edited])

#### post-edit

In [ ]:
flight_df7 = flight_df7.copy()
flight_df7.loc[flight_df7['FlightID'] == 1404001, 'FlightID'] = 1404001.1
new_row = flight_df7[flight_df7['FlightID'] == 1404001.1].copy()
new_row.loc[:, 'FlightID'] = 1404001.2
flight_df7 = pd.concat([flight_df7, new_row], ignore_index=True)
flight_df7['FlightID'].unique()

#ger rid of excluded flightIDs from flight_df7
flight_df7 = flight_df7[~flight_df7['FlightID'].isin([1602001, 1602002, 1602003, 1602004, 
                      1602401, 1602402, 1602403, 1602404, 1602405, 
                      1602801, 1602802, 1603201, 1603202, 1603203, 1603204,
                      1603601, 1603602, 1603603, 1603604, 1603605, 
                      2202804])]

In [ ]:
helpers_BW.plot_flight_data(flight_df7, df)

In [ ]:
df[df['FlightID'] == 1404001.2]

### chunk 8 

#### pre-edit

In [ ]:
helpers_BW.plot_flight_data(flight_df8, df)

In [ ]:
# for each flightID in df3_to_edit, make a subset with the correct time range
print(df.shape)
# make a list of flight ID to filter out 
flights_to_exclude = [2902803, 2902805, 2903201, 2903204, 2903604, 2903605, 2803202, 2803206,
                      2803207, 2803601, 2803603 
                      ]
#delete entirely 
# save the flights to edit first
df8_to_edit = df[df['FlightID'].isin(flights_to_exclude)]

# exclude from original dataset 
df = df[~df['FlightID'].isin(flights_to_exclude)
                        ]
print(df.shape) 


for flightID in df8_to_edit['FlightID'].unique():
    #21 flightIDs to exclude so only concatenating flights that are being trimmed

    if flightID == 2902803:
        subset1 = df8_to_edit[(df8_to_edit['FlightID'] == flightID) & (df8_to_edit['Time'] >= 10.0) &  (df8_to_edit['Time'] <= 14.0)]

    elif flightID == 2902805:
        subset2 = df8_to_edit[(df8_to_edit['FlightID'] == flightID) &  (df8_to_edit['Time'] <= 2.0)]
        subset2.loc[:, 'FlightID'] = 2902805.1
        subset3 = df8_to_edit[(df8_to_edit['FlightID'] == flightID) & (df8_to_edit['Time'] >= 10.0) & (df8_to_edit['Time'] <= 14.0)]
        subset3.loc[:, 'FlightID'] = 2902805.2
        
    elif flightID == 2903201:
       continue

    elif flightID == 2903204:
        subset4 = df8_to_edit[(df8_to_edit['FlightID'] == flightID) & (df8_to_edit['Time'] >= 2.0) & (df8_to_edit['Time'] <= 4.0)]

    elif flightID == 2903604:
        subset5 = df8_to_edit[(df8_to_edit['FlightID'] == flightID) & (df8_to_edit['Time'] >= 0.5) & (df8_to_edit['Time'] <= 2.5)]

    elif flightID == 2903605:

        subset6 = df8_to_edit[(df8_to_edit['FlightID'] == flightID) & (df8_to_edit['Time'] <= 5.0)]

    elif flightID == 2803202:
        subset7 = df8_to_edit[(df8_to_edit['FlightID'] == flightID) & (df8_to_edit['Time'] >= 1.0) & (df8_to_edit['Time'] <= 4.0)]

    elif flightID == 2803206:
        subset8 = df8_to_edit[(df8_to_edit['FlightID'] == flightID) & (df8_to_edit['Time'] >= 2.0) & (df8_to_edit['Time'] <= 4.0)]

    elif flightID == 2803207:
        subset9 = df8_to_edit[(df8_to_edit['FlightID'] == flightID) & (df8_to_edit['Time'] >= 2.0) & (df8_to_edit['Time'] <= 4.0)]

    elif flightID == 2803601:
        subset10 = df8_to_edit[(df8_to_edit['FlightID'] == flightID) & (df8_to_edit['Time'] >= 4.0) & (df8_to_edit['Time'] <= 5.0)]

    elif flightID == 2803603:
        subset11 = df8_to_edit[(df8_to_edit['FlightID'] == flightID) & (df8_to_edit['Time'] >= 4.0) & (df8_to_edit['Time'] <= 5.0)]

# concatenate all the subsets
df8_edited = pd.concat([subset1, subset2, subset3, subset4, subset5, subset6, subset7, subset8, subset9, subset10, subset11])

# concatenate back to flight_df8
df = pd.concat([df, df8_edited])

#### post-edit

In [ ]:
flight_df8 = flight_df8.copy()
flight_df8.loc[flight_df8['FlightID'] == 2902805, 'FlightID'] = 2902805.1
new_row = flight_df8[flight_df8['FlightID'] == 2902805.1].copy()
new_row.loc[:, 'FlightID'] = 2902805.2
flight_df8 = pd.concat([flight_df8, new_row], ignore_index=True)
flight_df8['FlightID'].unique()

#ger rid of excluded flightIDs from flight_df8
flight_df8 = flight_df8[~flight_df8['FlightID'].isin([2903201])]

In [ ]:
# had to be after 5 seconds not before 5 seconds. 
df8edit = df8_to_edit[(df8_to_edit['FlightID'] == 2903605) & (df8_to_edit['Time'] >= 5.0)]
df = df[df['FlightID']!= 2903605]
df = pd.concat([df, df8edit])

In [ ]:
helpers_BW.plot_flight_data(flight_df8, df)

### chunk 9 

#### pre-edit

In [ ]:
helpers_BW.plot_flight_data(flight_df9, df)

In [ ]:
print(df[df['FlightID'] == 3410001].shape)
print(df[df['FlightID'] == 3410002].shape)
print(df[df['FlightID'] == 3410003].shape)
print(df[df['FlightID'] == 3410004].shape)
print(df[df['FlightID'] == 3410005].shape)
print(df[df['FlightID'] == 3410006].shape)

In [ ]:
df[df['FlightID'] == 3410001]

In [ ]:
print(df.shape)
# make a list of flight ID to filter out 
flights_to_exclude = [3410001, 3410002, 3410003, 3410004, 3410005, 3410006,
                      2803604, 2803605, 2803606, 2803607, 2804001, 3402806 ]
#delete entirely 
# save the flights to edit first

df9_to_edit = df[df['FlightID'].isin(flights_to_exclude)]

# exclude from original dataset 
df = df[~df['FlightID'].isin(flights_to_exclude)
                        ]

print(df.shape) 
df9_to_edit.shape
# for each flightID in df9_to_edit, make a subset with the correct time range

for flightID in df9_to_edit['FlightID'].unique():

    if flightID == 3402806:
        continue

    elif flightID == 2803604:
        subset1 = df9_to_edit[(df9_to_edit['FlightID'] == flightID) & (df9_to_edit['Time'] >= 2.0) & (df9_to_edit['Time'] <= 6.0)]
    elif flightID == 2803605:
        subset2 = df9_to_edit[(df9_to_edit['FlightID'] == flightID) & (df9_to_edit['Time'] >= 2.0) & (df9_to_edit['Time'] <= 6.0)]
    elif flightID == 2803606:
        subset3 = df9_to_edit[(df9_to_edit['FlightID'] == flightID) & (df9_to_edit['Time'] >= 2.0) & (df9_to_edit['Time'] <= 6.0)]
    elif flightID == 2803607:
        subset4 = df9_to_edit[(df9_to_edit['FlightID'] == flightID) & (df9_to_edit['Time'] >= 2.0) & (df9_to_edit['Time'] <= 6.0)]

    elif flightID == 2804001:
        subset5 = df9_to_edit[(df9_to_edit['FlightID'] == flightID) & (df9_to_edit['Time'] >= 0.5) & (df9_to_edit['Time'] <= 2.0)]
       
                              
# concatenate all the subsets
df9_edited = pd.concat([subset1, subset2, subset3, subset4, subset5])

# concatenate back to flight_df4
df = pd.concat([df, df9_edited])

In [ ]:
flight_df9 = flight_df9[~flight_df9['FlightID'].isin([3410001, 3410002, 3410003, 3410004, 3410005, 3410006,3402806])]   

#### post-edit

In [ ]:
helpers_BW.plot_flight_data(flight_df9, df)

### chunk 10 

#### pre-edit

In [ ]:
helpers_BW.plot_flight_data(flight_df10, df)

In [ ]:
# for each flightID in df3_to_edit, make a subset with the correct time range
print(df.shape)
# make a list of flight ID to filter out 
flights_to_exclude = [3102807, 3410007, 3504002, 3504003]
                      
#delete entirely 
# save the flights to edit first
df10_to_edit = df[df['FlightID'].isin(flights_to_exclude)]

# exclude from original dataset 
df = df[~df['FlightID'].isin(flights_to_exclude)
                        ]
print(df.shape) 


for flightID in df10_to_edit['FlightID'].unique():
    #21 flightIDs to exclude so only concatenating flights that are being trimmed

    if flightID == 3102807 or flightID == 3410007:
        continue

    elif flightID == 3504002:
        subset1 = df10_to_edit[(df10_to_edit['FlightID'] == flightID) &  (df10_to_edit['Time'] >= -1.5) & (df10_to_edit['Time'] <= 0.5)]
        
        
    elif flightID == 3504003:
        subset2 = df10_to_edit[(df10_to_edit['FlightID'] == flightID) & (df10_to_edit['Time'] >= -1.0) & (df10_to_edit['Time'] <= 0.5)]

   
# concatenate all the subsets
df10_edited = pd.concat([subset1, subset2])

# concatenate back to flight_df8
df = pd.concat([df, df10_edited])

#### post-edit

In [ ]:
helpers_BW.plot_flight_data(flight_df10, df) 

### chunk 11

#### pre-edit

In [ ]:
helpers_BW.plot_flight_data(flight_df11, df)

#### just deleting 3103603

In [ ]:
df = df[df['FlightID'] != 3103603]
helpers_BW.plot_flight_data(flight_df11, df)

## 4. SAVE

In [ ]:
df.to_csv("C:\\Users\\kmh\\Documents\\DATA\\zebras_2025\\df_all_chunks_edited_final.csv")

# II. Head marker

## load the dataset

In [ ]:
takeoffs = pd.read_csv('../data/takeoff/takeoffs_2025.csv', index_col = 0, 
                dtype =  {'Bird': str,'Age': int,'Takeoff':str, 'Note':str, 'Marker': int, 'Time':float, 'Frame': int, 'X': float,'Y': float,'Z': float} )

In [ ]:
takeoffs.head(5)

In [ ]:
# randomly choose 10 unique combinations of Bird, Age, Takeoff
unique_flights = takeoffs[['Bird', 'Age', 'Takeoff']].drop_duplicates().sample(n=10, random_state=79)
unique_flights

## first look with 13 trials

##### yellowR-purpleL, age20, trial3: 

- there can be three markers with high no.of stationary labels
- can choose head marker using Y min value. The one with more positive value is head. 
- last bit of a chosen marker was dropped
- birds may not be measured in the same interval. 
- worked!
- think it worked? a bit disconnected. but what if all the markers are tracked? 

In [ ]:
takeoffs.info()

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(3, 1, figsize=(10, 12), sharex=True)

axs[0].scatter(current_sequence['Frame'], current_sequence['X'], color='r')
axs[0].set_ylabel('X Position')
axs[0].set_title('X vs Frame')

axs[1].scatter(current_sequence['Frame'], current_sequence['Y'], color='g')
axs[1].set_ylabel('Y Position')
axs[1].set_title('Y vs Frame')

axs[2].scatter(current_sequence['Frame'], current_sequence['Z'], color='b')
axs[2].set_ylabel('Z Position')
axs[2].set_title('Z vs Frame')
axs[2].set_xlabel('Frame')

plt.tight_layout()
plt.show()


In [ ]:
# no problem. marker is 3. 
current_sequence = Tracking.plot_markers(takeoffs, 'yellowR-purpleL', 20, "3.0")
head_sequence = Tracking.process_all_combinations(current_sequence)
head_sequence = Tracking.plot_markers(head_sequence, 'yellowR-purpleL', 20, "3.0")

- updated with comments below


- hmmm I think the head marker is actually 3 with the least Euclidean distance and start_Y == min_Y, and neck marker 2. 
- but can't prioritise Euclidean distance because in case of greenR-age32-trial3,the chosen head marker has higher Euclidean distance. But it's start_Y makes sense in relation to others. 
- sometimes start_Y != min_Y because of the ghost markers somewhere lower than the perch. 
- but start_Y is quite important because it gets the very first acceleration, to the point that it is almost meaningless as a head marker if it haven't got the first.



##### pinkRblackL, age40, trial 1
- In current code that prirotises start_Y more than anything, marker 3 cannot be head marker. 
- marker 3 could be a neck marker, but way more useful in tracking and should contain all the information on the bird's movement 
- **should add a section in the code where if start_Y difference is within 20 mm the Euclidean distance is checked**
- **But how much Euclidean difference is significantly different?**

- ok looking at this trial it makes sense to choose a marker with less standard deviation because look at marker 5. Looks like it could be a head marker but all over the place. also makes sense not using mean because marker 5' Y mean will be higher than marker 3's Y mean. Not using X because when the bird is compressed and in the moment of applying maximum resultant force, head marker will be definitely ahead of the neck, whereas X horizontal distance could be almost similar (2025)

- made identify_head_marker to choose head marker from two markers with highest stationary label values
- second marker dropped if the difference in the stationary value is more than 3. 
- changed the marker choice selection from .min to .std. check all the trials in this bird & age and all the previous birds

In [ ]:
# the head marker is 3. 
current_sequence = Tracking.plot_markers(takeoffs, 'pinkR-blackL', 40, "1.0")
head_sequence = Tracking.process_all_combinations(current_sequence)
head_sequence = Tracking.plot_markers(head_sequence, 'pinkR-blackL', 40, "1.0")

In [ ]:
Tracking.plot_single_marker(current_sequence, 5)

##### pinkR-blueL, age80, trial 2
- works - marker 1 at starting position and less Euclidean distance. 
- marker 2 (green) also looked promising but probably was lower in Y.min() value. 
- age100, trial 7: works with z-score threshold 2.4 

In [ ]:
current_sequence = Tracking.plot_markers(takeoffs, 'pinkR-blueL', 80, "2.0")
head_sequence = Tracking.process_all_combinations(current_sequence)
head_sequence = Tracking.plot_markers(head_sequence, 'pinkR-blueL', 80, "2.0")

In [ ]:
unique_flights


##### greenR-blueL, age 20, trial 3

- changed the Euclidean threshold to 50% -> 70% but still cannot make marker 5 a head marker because marker 1 is 6 cm more front than marker 2. 

In [ ]:
current_sequence = Tracking.plot_markers(takeoffs, 'greenR-blueL', 20, "3.0")
head_sequence = Tracking.process_all_combinations(current_sequence)
head_sequence = Tracking.plot_markers(head_sequence, 'greenR-blueL', 20, "3.0")

In [ ]:
Tracking.plot_single_marker(current_sequence, 5)

##### pinkR-blueL, age 80, trial 1
- head, neck, tail all well tracked 

In [ ]:
current_sequence = Tracking.plot_markers(takeoffs, 'pinkR-blueL', 80, "1.0")
head_sequence = Tracking.process_all_combinations(current_sequence)
head_sequence = Tracking.plot_markers(head_sequence, 'pinkR-blueL', 80, "1.0")

##### greenR, age 32, trial 3
- tricky - marker 6 looks like a start marker. Has a reasonably forward starting Y, but has a big Euclidean distance. 
- I think I'll just have to run all the markers once, and check the ones where the head marker doesn't look so good. 

- hmmm it might be a correct head marker for parts of it but there are a lot of ghost markers. 
- how about 6 and 7?
- Ideally the head marker is truly a head marker, but if the tracking is more inconsistent than neck marker, it's better to track the neck marker as long as that represents the bird. 
- priorities now: start_Y most forefront. but if the next start_Y is within 20 mm difference and has less Euc distance, that's better to track. 

In [ ]:
current_sequence = Tracking.plot_markers(takeoffs, 'greenR', 32, "3.0")
head_sequence = Tracking.process_all_combinations(current_sequence)
head_sequence = Tracking.plot_markers(head_sequence, 'greenR', 32, "3.0")

In [ ]:
Tracking.plot_single_marker(current_sequence, 3)
Tracking.plot_single_marker(current_sequence, 6)




- hmmm seems like marker 6 is the real head marker. 

##### pinkR-blueL, age 36, trial 7

In [ ]:
current_sequence = Tracking.plot_markers(takeoffs, 'pinkR-blueL', 36, "7.0")
head_sequence = Tracking.process_all_combinations(current_sequence)
head_sequence = Tracking.plot_markers(head_sequence, 'pinkR-blueL', 36, "7.0")

##### greenR-whiteL. age 80, trial 2 

In [ ]:
current_sequence = Tracking.plot_markers(takeoffs, 'greenR-whiteL', 80, "2.0")
head_sequence = Tracking.process_all_combinations(current_sequence)
head_sequence = Tracking.plot_markers(head_sequence, 'greenR-whiteL', 80, "2.0")

In [ ]:
current_sequence  

##### greenR-blueL, age 28, trial 5 

In [ ]:
current_sequence = Tracking.plot_markers(takeoffs, 'greenR-blueL', 28, "5.0")
head_sequence = Tracking.process_all_combinations(current_sequence)
head_sequence = Tracking.plot_markers(head_sequence, 'greenR-blueL', 28, "5.0")

##### pinkR-blackL, age 28, trial 5

In [ ]:
current_sequence = Tracking.plot_markers(takeoffs, 'pinkR-blackL', 80, "5.0")
head_sequence = Tracking.process_all_combinations(current_sequence)
head_sequence = Tracking.plot_markers(head_sequence, 'pinkR-blackL', 80, "5.0")

##### purpleR-blackL, age 100, trial 2

In [ ]:
current_sequence = Tracking.plot_markers(takeoffs, 'purpleR-blackL', 100, "2.0")
head_sequence = Tracking.process_all_combinations(current_sequence)
head_sequence = Tracking.plot_markers(head_sequence, 'purpleR-blackL', 100, "2.0")

##### yellowL, age 60, trial 4 

In [ ]:
current_sequence = Tracking.plot_markers(takeoffs, 'yellowL', 60, "4.0")
head_sequence = Tracking.process_all_combinations(current_sequence)
head_sequence = Tracking.plot_markers(head_sequence, 'yellowL', 60, "4.0")

##### greenR-orangeL, age 28, trial 1 

In [ ]:
current_sequence = Tracking.plot_markers(takeoffs, 'greenR-orangeL', 80, "5.0")
head_sequence = Tracking.process_all_combinations(current_sequence)
head_sequence = Tracking.plot_markers(head_sequence, 'greenR-orangeL', 80, "5.0")

## Protocol 
- Run all the trials with current logic
- Review all the outcomes in 10 chunks of 4 columns x 25 rows plot 
    - 11th chunk just 4 columns x 2 rows 
- For each chunk, re-check all the markers, and if the head marker needs changing, write it in the log
- delete that chunk from the 'outcome', and replace it with a new head sequence with the different marker. 

## 1. Load the traj_df

In [ ]:
traj_df = pd.read_csv("../data/takeoff/processed_df.csv", index_col =0)

In [ ]:
traj_df.head(5)

## 2. Make 10 chunks of 100 flights 

## 3. traj_ids for FlightID

In [ ]:
force_traj = pd.read_csv("../data/takeoff/force_df_for_traj.csv")

In [ ]:
force_traj.head(5)

In [ ]:
force_ids = helpers_BW.make_flight_df(force_traj)

In [ ]:
force_ids

In [ ]:
traj_df_ids = traj_df[['Bird','Age','Takeoff']].drop_duplicates().reset_index(drop=True )
traj_df_ids 

In [ ]:
# inner merge with Force on 'FlightID'
# does this leave 'FlightID' na if there is no matching 'FlightID' from force_ids?
traj_ids = pd.merge(traj_df_ids, force_ids, on=['Bird','Age','Takeoff'], how='left')
traj_ids


In [ ]:
traj_ids.info()

In [ ]:
#show the ones where 'FlightID' is na
print(traj_ids[traj_ids['FlightID'].isna()])

#also show traj_ids where 'Bird' == 'pinkR-blueL' and 'Age' == 40 and 'Takeoff' == '1.0'
print(traj_ids[(traj_ids['Bird'] == 'pinkR-blueL') & (traj_ids['Age'] == 40) & (traj_ids['Takeoff'] == 1.0)])

#show traj_ids where 'Bird' == 'redR-purpleL' and 'Age' == 28 and 'Takeoff' == '5.0'
print(traj_ids[(traj_ids['Bird'] == 'redR-purpleL') & (traj_ids['Age'] == 28) & (traj_ids['Takeoff'] == 5.0)])

# where 'FlightID' == 1404001.1, change 'Takeoff' to '0.0'
traj_ids.loc[traj_ids['FlightID'] == 1404001.1, 'Takeoff'] = 0.0

# where 'FlightID' == 2902805.1, change 'Takeoff' to '6.0'
traj_ids.loc[traj_ids['FlightID'] == 2902805.2, 'Takeoff'] = 6.0

In [ ]:
print(traj_ids[traj_ids['FlightID'].isna()])

# confirmed that 'pinkR-blueL', 40, 0.0 is now present.
print(traj_ids[(traj_ids['Bird'] == 'pinkR-blueL') & (traj_ids['Age'] == 40) & (traj_ids['Takeoff'] == 0.0)])

#drop where 'FlightID' is na
traj_ids = traj_ids.dropna(subset=['FlightID'])

In [ ]:
#to put FlightID in traj_df
traj_df = pd.merge(traj_df, traj_ids, on=['Bird','Age','Takeoff'], how='left')
traj_df.head(5)

In [ ]:
traj_df['FlightID'].nunique() #994->964->997

In [ ]:
traj_df[traj_df['FlightID'].isna()]

In [ ]:
# Make 10 chunks of 100 tracks
track_df1 = traj_ids.iloc[0:100,:]
track_df2 = traj_ids.iloc[100:200,:]
track_df3 = traj_ids.iloc[200:300,:]
track_df4 = traj_ids.iloc[300:400,:]
track_df5 = traj_ids.iloc[400:500,:]
track_df6 = traj_ids.iloc[500:600,:]
track_df7 = traj_ids.iloc[600:700,:]
track_df8 = traj_ids.iloc[700:800,:]
track_df9 = traj_ids.iloc[800:900,:]
track_df10 = traj_ids.iloc[900:,:]

## 4. chunk reviews

### chunk 1

In [ ]:
Tracking.plot_flight_trajectory_grid(track_df1, traj_df)

### chunk 2

In [ ]:
Tracking.plot_flight_trajectory_grid(track_df2, traj_df)

### chunk 3

In [ ]:
Tracking.plot_flight_trajectory_grid(track_df3, traj_df)

### chunk 4

In [ ]:
Tracking.plot_flight_trajectory_grid(track_df4, traj_df)

### chunk 5

In [ ]:
Tracking.plot_flight_trajectory_grid(track_df5, traj_df)

- chunk 2-5 for some reason deleted??

### chunk 6

In [ ]:
Tracking.plot_flight_trajectory_grid(track_df6, traj_df)

### chunk 7

In [ ]:
Tracking.plot_flight_trajectory_grid(track_df7, traj_df)

### chunk 8 

In [ ]:
Tracking.plot_flight_trajectory_grid(track_df8, traj_df)

### chunk 9

In [ ]:
Tracking.plot_flight_trajectory_grid(track_df9, traj_df)

### chunk 10

In [ ]:
Tracking.plot_flight_trajectory_grid(track_df10, traj_df)

# III. Reviewing 

## load takeoffs and processed_df

In [ ]:
takeoffs = pd.read_csv('../data/takeoff/takeoffs_2025.csv', index_col = 0, 
                dtype =  {'Bird': str,'Age': int,'Takeoff':int, 'Note':str, 'Marker': int, 'Time':float, 'Frame': int, 'X': float,'Y': float,'Z': float} )
takeoffs.head(5)

In [ ]:
takeoffs.info()

In [ ]:
processed_df_final = pd.read_csv("../data/takeoff/processed_df_final.csv", index_col=0, dtype =  {'Bird': str,'Age': int,'Takeoff':int, 'Note':str, 'Marker': int, 'Time':float, 'Frame': int, 'X': float,'Y': float,'Z': float})
processed_df_final.head(5)

In [ ]:
processed_df_final.info()

In [ ]:
review_df = pd.read_csv("../data/takeoff/flights_to_review.csv")
review_df = review_df[['Chunk','FlightID','Bird','Age','Trial','Note']]
review_df

In [ ]:
review_chunk1.info()

### dividing in chunks of 20

In [ ]:
review_chunk1 = review_df.iloc[0:20, :]
review_chunk2 = review_df.iloc[20:40, :]
review_chunk3 = review_df.iloc[40:60, :]
review_chunk4 = review_df.iloc[60:80, :]
review_chunk5 = review_df.iloc[80:100, :]
review_chunk6 = review_df.iloc[100:120, :]
review_chunk7 = review_df.iloc[120:140, :]
review_chunk8 = review_df.iloc[140:, :]




In [ ]:
review_chunk1.info

## review_chunk1

In [ ]:
Tracking.review_flights(review_chunk1, takeoffs, processed_df_final)

## review_chunk2

In [ ]:
Tracking.review_flights(review_chunk2, takeoffs, processed_df_final)

## review_chunk3

In [ ]:
Tracking.review_flights(review_chunk3, takeoffs, processed_df_final)

## review_chunk4

In [ ]:
Tracking.review_flights(review_chunk4, takeoffs, processed_df_final)

## review_chunk5

In [ ]:
Tracking.review_flights(review_chunk5, takeoffs, processed_df_final)

## review_chunk6

In [ ]:
Tracking.review_flights(review_chunk6, takeoffs, processed_df_final)

## review_chunk7

In [ ]:
Tracking.review_flights(review_chunk7, takeoffs, processed_df_final)

## review_chunk8

In [ ]:
Tracking.review_flights(review_chunk8, takeoffs, processed_df_final)

# IV. Editing

## attaching flight ID to takeoffs and processed_df_final

In [ ]:
traj_ids['Takeoff'] = traj_ids['Takeoff'].astype(int)
traj_ids

In [ ]:
# merge to processed_df_final - merge just once
processed_df_final = pd.merge(processed_df_final, traj_ids, on = ["Bird", "Age", "Takeoff"], how = "left")
takeoffs = pd.merge(takeoffs, traj_ids, on = ["Bird", "Age", "Takeoff"], how = "left")


## Review chunk 1

In [ ]:
print(processed_df_final)
print(takeoffs)

#both now have FlightID

In [ ]:
#make a function that displays diagnostics ('Stationary Labels' and all the markers overlayed ) and X,Y, Z of all the markers. 
# input -> flightID and takeoffs 
# clip the needed marker and save as chunk_edited1

takeoffs

In [ ]:
takeoffs.dtypes

### edited_2310001, 2310009, 2102404, 2103210, 2103603

In [ ]:
edited_2310001 = takeoffs[(takeoffs['FlightID'] == 2310001.0) &(takeoffs['Marker'] == 2) & (takeoffs['Z'] > 1325)]
edited_2310009 = takeoffs[(takeoffs['FlightID'] == 2310009.0) & (takeoffs['Marker'] == 3)]
edited_2102404 = takeoffs[(takeoffs['FlightID'] == 2102404.0) & (takeoffs['Marker'] == 7)]
edited_2103210 = takeoffs[(takeoffs['FlightID'] == 2103210.0) & (takeoffs['Marker'] ==2) & (takeoffs['Z']< 1400)]
edited_2103603 = takeoffs[(takeoffs['FlightID'] == 2103603.0) & (takeoffs['Marker'] == 1)]

### Flight ID == 2102801

In [ ]:
Tracking.review_individual_flight(2102801, takeoffs, processed_df_final)

#### edited_2102801

In [ ]:
edited_2102801 = takeoffs[(takeoffs['FlightID'] == 2102801) & (takeoffs['Marker'] == 2)].iloc[:-15]

In [ ]:
takeoffs[takeoffs['FlightID'] == 2102802].head(5)

In [ ]:
processed_df_final[processed_df_final['FlightID'] == 2102802].head(5)

### Flight ID == 2102802 -> drop. 

In [ ]:
Tracking.review_individual_flight(2102802, takeoffs, processed_df_final)

In [ ]:
Tracking.review_individual_flight(2103203, takeoffs, processed_df_final)

### edited_2103203

In [ ]:
edited_2103203 = takeoffs[(takeoffs['FlightID'] == 2103203) & (takeoffs['Marker'] == 4)]
edited_2103203.shape

In [ ]:
edited_2103203 = takeoffs[(takeoffs['FlightID'] == 2103203) & (takeoffs['Marker'] == 4)].iloc[:-7]

In [ ]:
edited_2103203.shape

In [ ]:
Tracking.review_individual_flight(2106001, takeoffs, processed_df_final)

### edited_2106001

In [ ]:
edited_2106001 = takeoffs[(takeoffs['FlightID'] == 2106001) & (takeoffs['Marker'] == 1)]
print(edited_2106001.shape)

edited_2106001 = takeoffs[(takeoffs['FlightID'] == 2106001) & (takeoffs['Marker'] == 1) & (takeoffs['Z'] > 800)]
print(edited_2106001.shape)

### filter out chunk to edit

In [ ]:
print(processed_df_final.shape) #53201
processed_df_final_except1 = processed_df_final[~processed_df_final['FlightID'].isin([2310001, 210009, 2102403, 2102404, 2102801, 2102802, 2103203, 2103210, 2103603, 2106001, 2106003])]
print(processed_df_final_except1) #dataframe to add the edited ones to. #52717

In [ ]:
print(processed_df_final_except1.shape)
processed_df_final_chunk1_done = pd.concat([processed_df_final_except1, edited_2310001, edited_2310009, edited_2102404, edited_2102801, edited_2103203, edited_2103210, edited_2103603, edited_2106001], axis = 0, ignore_index = True)
print(processed_df_final_chunk1_done.shape)

In [ ]:
processed_df_final_chunk1_done

## Review chunk 2

### edited_3702004, 3706007, 3706011

In [ ]:
edited_3702004 = takeoffs[(takeoffs['FlightID'] == 3702004) & (takeoffs['Marker'] == 1)]
edited_3706007 = takeoffs[(takeoffs['FlightID'] == 3706007) & (takeoffs['Marker'] == 4)]
edited_3706011 = takeoffs[(takeoffs['FlightID'] == 3706011) & (takeoffs['Marker'] == 4)]


### filter out chunks to edit

In [ ]:
print(processed_df_final_chunk1_done.shape) #53100
processed_df_final_except2 = processed_df_final_chunk1_done[~processed_df_final_chunk1_done['FlightID'].isin([2106004, 2106005, 2106006, 3702004, 3706007, 3706011, 2703207])]
print(processed_df_final_except2) #dataframe to add the edited ones to. #52795

### concat

In [ ]:
processed_df_final_chunk12_done = pd.concat([processed_df_final_except2, edited_3702004, edited_3706007, edited_3706011])
print(processed_df_final_chunk12_done.shape) #52975

## Review chunk 3 

In [ ]:
takeoffs.dtypes

### edited_2710001, 2710002, 2710004, 2710006, 2710008, 2710011, 2710012, 2602003, 2610002, 2610003, 2610006, 2610007

In [ ]:
edited_2710001 = takeoffs[(takeoffs['FlightID'] == 2710001) & (takeoffs['Marker'] == 2) & (takeoffs['Z'] > 800)].iloc[:-1]
edited_2710002 = takeoffs[(takeoffs['FlightID'] == 2710002) & (takeoffs['Marker'] == 3) & (takeoffs['X'] > 100)]
edited_2710004 = takeoffs[(takeoffs['FlightID'] == 2710004) & (takeoffs['Marker'] == 5) & (takeoffs['Z'] > 1325)]
edited_2710006 = takeoffs[(takeoffs['FlightID'] == 2710006) & (takeoffs['Marker'] == 3) & (takeoffs['Z'] > 1000).iloc[:-1]]
edited_2710008 = takeoffs[(takeoffs['FlightID'] == 2710008) & (takeoffs['Marker'] == 9)].iloc[:-8]
edited_2710011 = takeoffs[(takeoffs['FlightID'] == 2710011) & (takeoffs['Marker'] == 4) & (takeoffs['Y'] > -470.3)]
edited_2710012 = takeoffs[(takeoffs['FlightID'] == 2710012) & (takeoffs['Marker'] == 4) & (takeoffs['Z'] > 1200)]
edited_2602003 = takeoffs[(takeoffs['FlightID'] == 2602003) & (takeoffs['Marker'] == 3)].iloc[:-16]
edited_2610002 = takeoffs[(takeoffs['FlightID'] == 2610002) & (takeoffs['Marker'] == 6) & (takeoffs['Y'] > -401.8)]
edited_2610003_a = takeoffs[(takeoffs['FlightID'] == 2610003) & (takeoffs['Marker'] == 9) & (takeoffs['Z'] > 1315) & (takeoffs['Time'] <= 2.3)]
edited_2610003_b = takeoffs[(takeoffs['FlightID'] == 2610003) & (takeoffs['Marker'] == 9) & (takeoffs['Time'] > 2.3)]
edited_2610006 = takeoffs[(takeoffs['FlightID'] == 2610006) & (takeoffs['Marker'] == 9)].iloc[:-4]
edited_2610007 = takeoffs[(takeoffs['FlightID'] == 2610007) & (takeoffs['Marker'] == 7) & (takeoffs['X'] > 100)]






### filter out chunk 3

In [ ]:
print(processed_df_final_chunk12_done.shape) #52975
processed_df_final_except3 = processed_df_final_chunk12_done[~processed_df_final_chunk12_done['FlightID'].isin([2710001, 2710002, 2710003, 2710004, 2710005, 2710006, 2710007, 
                                                                                                                2710008, 2710009, 2710010, 2710011, 2710012, 2602003, 2610002, 
                                                                                                                2610003, 2610006, 2610007])]
print(processed_df_final_except3.shape) #52290

### concat

In [ ]:
processed_df_final_chunk123_done = pd.concat([processed_df_final_except3, edited_2710001, edited_2710002, edited_2710004, edited_2710006, edited_2710008, edited_2710011, 
                                                                          edited_2710012, edited_2602003, edited_2610002, edited_2610003_a, edited_2610003_b, edited_2610006, edited_2610007])
print(processed_df_final_chunk123_done.shape) #52673

In [ ]:
Tracking.review_individual_flight(2610007, takeoffs, processed_df_final)

## Review chunk 4

### edited_2610008, 2610009, 3003602, 1302401, 1303203, 1303205, 1303206

In [ ]:
edited_2610008 = takeoffs[(takeoffs['FlightID'] == 2610008) & (takeoffs['Marker'] == 3) & (takeoffs['Y'] > -451.3)]
edited_2610009 = takeoffs[(takeoffs['FlightID'] == 2610009) & (takeoffs['Marker'] == 4) & (takeoffs['Y'] > -475.9)]
edited_3003602 = takeoffs[(takeoffs['FlightID'] == 3003602) & (takeoffs['Marker'] == 4)]
edited_1302401 = takeoffs[(takeoffs['FlightID'] == 1302401) & (takeoffs['Marker'] == 0)]
edited_1303203 = takeoffs[(takeoffs['FlightID'] == 1303203) & (takeoffs['Marker'] == 1)]
edited_1303205 = takeoffs[(takeoffs['FlightID'] == 1303205) & (takeoffs['Marker'] == 3)]
edited_1303206 = takeoffs[(takeoffs['FlightID'] == 1303206) & (takeoffs['Marker'] == 1)].iloc[:-6]


### filter out chunk 4

In [ ]:
print(processed_df_final_chunk123_done.shape) #52975
processed_df_final_except4 = processed_df_final_chunk123_done[~processed_df_final_chunk123_done['FlightID'].isin([2610008, 2610009, 3002401, 3002402, 3003602, 3010001, 1302401, 1303203, 1303205, 1303206])]
print(processed_df_final_except4.shape) #52222

### concat

In [ ]:
processed_df_final_chunk1234_done = pd.concat([processed_df_final_except4, edited_2610008, edited_2610009, edited_3003602, edited_1302401, edited_1303203, edited_1303205, edited_1303206]) #7/10
print(processed_df_final_chunk1234_done.shape) #52559


In [ ]:
processed_df_final_chunk1234_done

In [ ]:
processed_df_final_chunk1234_done.to_csv('../data/takeoff/processed_df_final_chunk1234_done.csv')

In [ ]:
Tracking.review_individual_flight(1303206, takeoffs, processed_df_final)

## Review chunk 5

### edited_1306003, 1306004, 1306005, 1306007, 1402405, 1402804, 1402805, 1403207, 1406001, 1406002, 1606007, 2902001

In [ ]:
edited_1306003 = takeoffs[(takeoffs['FlightID'] == 1306003) & (takeoffs['Marker'] == 7) & (takeoffs['Z'] > 800)]
edited_1306005 = takeoffs[(takeoffs['FlightID'] == 1306005) & (takeoffs['Marker'] == 3) & (takeoffs['Z'] > 750)]
edited_1306007 = takeoffs[(takeoffs['FlightID'] == 1306007) & (takeoffs['Marker'] == 2)]
edited_1402804 = takeoffs[(takeoffs['FlightID'] == 1402804) & (takeoffs['Marker'] == 1)]
edited_1402805 = takeoffs[(takeoffs['FlightID'] == 1402805) & (takeoffs['Marker'] == 2)]
edited_1403207 = takeoffs[(takeoffs['FlightID'] == 1403207) & (takeoffs['Marker'] == 2)]
edited_1406001 = takeoffs[(takeoffs['FlightID'] == 1406001) & (takeoffs['Marker'] == 2) & (takeoffs['Z'] > 1450)]
edited_1406002 = takeoffs[(takeoffs['FlightID'] == 1406002) & (takeoffs['Marker'] == 2) & (takeoffs['Z'] > 1500)]
edited_1606007 = takeoffs[(takeoffs['FlightID'] == 1606007) & (takeoffs['Marker'] == 1) & (takeoffs['Z'] > 800)]


### filter out chunk 5 

In [ ]:
processed_df_final_chunk1234_done = pd.read_csv('../data/takeoff/processed_df_final_chunk1234_done.csv', index_col = 0)
processed_df_final_chunk1234_done.head(5)

In [ ]:
print(processed_df_final_chunk1234_done.shape) #52559
processed_df_final_except5 = processed_df_final_chunk1234_done[~processed_df_final_chunk1234_done['FlightID'].isin([1306003, 1306004, 1306005, 1306007, 1402804, 1402805, 1403207, 1406001, 1406002, 1606007, 2902001])]
print(processed_df_final_except5.shape) #52142

### concat

In [ ]:
processed_df_final_chunk12345_done = pd.concat([processed_df_final_except5, edited_1306003, edited_1306005, edited_1306007, edited_1402804, edited_1402805, 
                                                                        edited_1403207, edited_1406001, edited_1406002, edited_1606007]) #9/11
print(processed_df_final_chunk12345_done.shape) #52513


In [ ]:
Tracking.review_individual_flight(1306003, takeoffs, processed_df_final)

## Review chunk 6

### edited_2902002, 2902005, 2902006, 2902007, 2903206, 2903207, 2904003, 2904004, 2904005, 2910005,

In [ ]:
edited_2902002 = takeoffs[(takeoffs['FlightID'] == 2902002) & (takeoffs['Marker'] == 7) & (takeoffs['X'] > 100)]
edited_2902005 = takeoffs[(takeoffs['FlightID'] == 2902005) & (takeoffs['Marker'] == 4)].iloc[:-15]
edited_2902006 = takeoffs[(takeoffs['FlightID'] == 2902006) & (takeoffs['Marker'] == 8) & (takeoffs['Time'] <= 4.3) & (takeoffs['Z'] > 1000)]
edited_2902007 = takeoffs[(takeoffs['FlightID'] == 2902007) & (takeoffs['Marker'] == 3)]
edited_2903206 = takeoffs[(takeoffs['FlightID'] == 2903206) & (takeoffs['Marker'] == 2)]
edited_2903207 = takeoffs[(takeoffs['FlightID'] == 2903207) & (takeoffs['Marker'] == 1)]
edited_2904003 = takeoffs[(takeoffs['FlightID'] == 2904003) & (takeoffs['Marker'] == 2)]
edited_2904004 = takeoffs[(takeoffs['FlightID'] == 2904004) & (takeoffs['Marker'] == 3) & (takeoffs['Z'] > 600)]
edited_2904005 = takeoffs[(takeoffs['FlightID'] == 2904005) & (takeoffs['Marker'] == 5)]
edited_2910005 = takeoffs[(takeoffs['FlightID'] == 2910005) & (takeoffs['Marker'] == 7)]




### filter out chunk 6

In [ ]:
print(processed_df_final_chunk12345_done.shape) #52513
processed_df_final_except6 = processed_df_final_chunk12345_done[~processed_df_final_chunk12345_done['FlightID'].isin([2902002, 2902005, 2902006, 2902007, 2903206, 2903207, 
                                                                                                                      2903607, 2904003, 2904004, 2904005, 2910005])]
print(processed_df_final_except6.shape) #52023

### concat

In [ ]:
processed_df_final_chunk123456_done = pd.concat([processed_df_final_except6, edited_2902002, edited_2902005, edited_2902006, edited_2902007, edited_2903206, edited_2903207, edited_2904003, edited_2904004, edited_2904005, edited_2910005]) #10/11
print(processed_df_final_chunk123456_done.shape) #52527

In [ ]:
processed_df_final_chunk123456_done.to_csv('../data/takeoff/processed_df_final_chunk123456_done.csv')

In [ ]:
Tracking.review_individual_flight(2910015, takeoffs, processed_df_final)

## Review chunk 7

In [ ]:
Tracking.review_individual_flight(3508007, takeoffs, processed_df_final)

### edited_

In [ ]:
edited_2802804 = takeoffs[(takeoffs['FlightID'] == 2802804) & (takeoffs['Marker'] == 1)].iloc[:-7]
edited_2810002 = takeoffs[(takeoffs['FlightID'] == 2810002) & (takeoffs['Marker'] == 7) & (takeoffs['Z'] > 1000)]
edited_2810003 = takeoffs[(takeoffs['FlightID'] == 2810003) & (takeoffs['Marker'] == 1) & (takeoffs['Z'] > 1000)]
edited_2810004 = takeoffs[(takeoffs['FlightID'] == 2810004) & (takeoffs['Marker'] == 4) & (takeoffs['Z'] > 1325)]
edited_3408017 = takeoffs[(takeoffs['FlightID'] == 3408017) & (takeoffs['Marker'] == 2)]
edited_3506002 = takeoffs[(takeoffs['FlightID'] == 3506002) & (takeoffs['Marker'] == 3) & (takeoffs['Z'] >1325)]
edited_3506004 = takeoffs[(takeoffs['FlightID'] == 3506004) & (takeoffs['Marker'] == 1)]
edited_3508004 = takeoffs[(takeoffs['FlightID'] == 3508004) & (takeoffs['Marker'] == 3)]
edited_3508007 = takeoffs[(takeoffs['FlightID'] == 3508007) & (takeoffs['Marker'] == 1)]


### filter out chunk 7

In [ ]:
print(processed_df_final_chunk123456_done.shape) #52527
processed_df_final_except7 = processed_df_final_chunk123456_done[~processed_df_final_chunk123456_done['FlightID'].isin([2802804, 2810001, 2810002, 2810003, 2810004, 
                                                                                                                        2810005, 3408017, 3502008, 3502009, 3502010, 
                                                                                                                        3506002, 3506004, 3508004, 3508007])]
print(processed_df_final_except7.shape) #51935

### concat

In [ ]:
processed_df_final_chunk1234567_done = pd.concat([processed_df_final_except7, edited_2802804, edited_2810002, edited_2810003, edited_2810004, edited_3408017, edited_3506002, edited_3506004, edited_3508004, edited_3508007]) # 9/14
print(processed_df_final_chunk1234567_done.shape) #52344

## Review chunk 8

### edited_

In [ ]:
edited_3510010 = takeoffs[(takeoffs['FlightID'] == 3510010) & (takeoffs['Marker'] == 4)]
edited_3102804 = takeoffs[(takeoffs['FlightID'] == 3102804) & (takeoffs['Marker'] == 7)]
edited_1206001 = takeoffs[(takeoffs['FlightID'] == 1206001) & (takeoffs['Marker'] == 1)]
edited_1206002 = takeoffs[(takeoffs['FlightID'] == 1206002) & (takeoffs['Marker'] == 2)]


In [ ]:
takeoffs[(takeoffs['Bird'] == 'redR-purpleL') & (takeoffs['Age'] == 28)]['FlightID'].unique()

### filter out chunk 8

In [ ]:
print(processed_df_final_chunk1234567_done.shape) #52344
processed_df_final_except8 = processed_df_final_chunk1234567_done[~processed_df_final_chunk1234567_done['FlightID'].isin([3510010, 3102804, 1206001, 1206002, 1404003, 1404001.2])]
print(processed_df_final_except8.shape) #52131

### concat

In [ ]:
processed_df_final_chunk12345678_done = pd.concat([processed_df_final_except8, edited_3510010, edited_3102804, edited_1206001, edited_1206002]) # 9/14
print(processed_df_final_chunk12345678_done.shape) #52318

In [ ]:
Tracking.review_individual_flight(1404001.2, takeoffs, processed_df_final)

# SAVE

In [ ]:
processed_df_final_chunk12345678_done.to_csv('../data/takeoff/clean_flights.csv')

# V. Clean Flights Final Look

In [ ]:
clean_flights = pd.read_csv('../data/takeoff/clean_flights.csv', index_col = 0)
clean_flights.head(10)

In [ ]:
print(f'no. of flight: {clean_flights['FlightID'].nunique()}') #973 flights 
print(f'no. of individuals: {clean_flights['Bird'].nunique()}') # 17 individuals